In [1]:
import json
import sys
import os
import pandas as pd
import random
import logging
import torch
from torch.utils.data import DataLoader

models_path = os.path.abspath(os.path.join('models'))
sys.path.append(models_path)

src_path = os.path.abspath(os.path.join('src'))
sys.path.append(src_path)

data_path = os.path.abspath(os.path.join('data'))
sys.path.append(data_path)

from data_loader_helper import SequenceDataset, collate_batch
from DKT.dkt_k_fold import k_fold_cv_dkt
from DKT.dkt_train import train_dkt
from KTDataset import KTDataset


In [4]:
df_answers = pd.read_csv('data/preprocessed/df_answers.csv')
df_skill_names = pd.read_csv('data/preprocessed/df_skill_names.csv')

kt_dataset = KTDataset(df_answers, df_skill_names, prepare_DKT=True)

user_dict = kt_dataset.DKT_datadict
num_skills = kt_dataset.num_skills
num_other = kt_dataset.num_other


# Set constants

In [ ]:
NUM_EPOCHS = 15
BATCH_SIZE = 100
NUM_FOLDS = 5  # Number of folds for cross-validation
HID_SIZE = 200

embed_dim = 2

train_ratio = 0.8  # 80% for training, 20% for testing

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Set configs

In [ ]:
configs = [
    {
        "learning_rate": lr,
        "models_params": {
            "num_skills": num_skills,
            "num_other": num_other,
            "embed_dim": embed_dim,
            "hid_size": HID_SIZE,
            "num_hid_layers": num_hid_layers,
            "drop_prob": drop_prob,
        },
    }
    for lr in [1e-3, 1e-4, 1e-5]  # 3 reasonable options for learning rate
    for num_hid_layers in [1, 2]  # Hidden layers 1 or 2
    for drop_prob in [0.3, 0.4, 0.5]  # Dropout rate 0.3, 0.4, 0.5
]

# Example: Printing configurations
for idx, config in enumerate(configs, 1):
    print(f"Config {idx}:\n{config}\n")



Config 1:
{'learning_rate': 0.001, 'models_params': {'num_skills': 109, 'num_other': 1, 'embed_dim': 2, 'hid_size': 200, 'num_hid_layers': 1, 'drop_prob': 0.3}}

Config 2:
{'learning_rate': 0.001, 'models_params': {'num_skills': 109, 'num_other': 1, 'embed_dim': 2, 'hid_size': 200, 'num_hid_layers': 1, 'drop_prob': 0.4}}

Config 3:
{'learning_rate': 0.001, 'models_params': {'num_skills': 109, 'num_other': 1, 'embed_dim': 2, 'hid_size': 200, 'num_hid_layers': 1, 'drop_prob': 0.5}}

Config 4:
{'learning_rate': 0.001, 'models_params': {'num_skills': 109, 'num_other': 1, 'embed_dim': 2, 'hid_size': 200, 'num_hid_layers': 2, 'drop_prob': 0.3}}

Config 5:
{'learning_rate': 0.001, 'models_params': {'num_skills': 109, 'num_other': 1, 'embed_dim': 2, 'hid_size': 200, 'num_hid_layers': 2, 'drop_prob': 0.4}}

Config 6:
{'learning_rate': 0.001, 'models_params': {'num_skills': 109, 'num_other': 1, 'embed_dim': 2, 'hid_size': 200, 'num_hid_layers': 2, 'drop_prob': 0.5}}

Config 7:
{'learning_rate': 

# Split data into train and test sets

In [ ]:

# Get all keys and shuffle them
keys = list(user_dict.keys())
random.shuffle(keys)

# Split keys into train and test
split_index = int(len(keys) * train_ratio)
train_keys = keys[:split_index]
test_keys = keys[split_index:]

# Create train and test dictionaries
train_dict = {key: user_dict[key] for key in train_keys}
test_dict = {key: user_dict[key] for key in test_keys}


Set up logging

In [ ]:
log_filename = f'model_evaluation_embed_dim_{embed_dim}.log'
logging.basicConfig(filename=log_filename, level=logging.INFO, 
                    format='%(asctime)s - %(message)s')


# Hyperparameter-tuning

In [ ]:
best_val_auc_avg = 0  # Track the best validation AUC
best_config = None  # Track the best configuration

for idx, config in enumerate(configs, 1):
    log_message = f"\nEvaluating Config {idx}/{len(configs)}"
    print(log_message)  # Print to console
    logging.info(log_message)  # Log to file

    lr = config['learning_rate']
    model_params = config['models_params']

    val_auc_avg = k_fold_cv_dkt(
        num_folds=NUM_FOLDS,
        model_params=model_params,
        lr=lr,
        num_epochs=NUM_EPOCHS,
        device=device,
        train_dict=train_dict,
        batch_size=BATCH_SIZE,
        )

    # Update the global best if needed
    if val_auc_avg > best_val_auc_avg:
        best_val_auc_avg = val_auc_avg
        best_config = config  # Save the best configuration
        best_message = f"\nNew best model found: Config {idx}. Validation AUC: {best_val_auc_avg:.4f}"
        print(best_message)  # Print to console
        logging.info(best_message)  # Log to file

# Final output
final_message = f"\nBest test AUC: {best_val_auc_avg:.4f}"
print(final_message)
logging.info(final_message)

final_config_message = f"\nBest Configuration: {best_config}"
print(final_config_message)
logging.info(final_config_message)


In [103]:
best_config = configs[0]

In [105]:
# Training on the whole train set
train_dict = dict(sorted(train_dict.items(), key=lambda item: len(item[1])))
train_dataset = SequenceDataset(train_dict)
train_loader = DataLoader(train_dataset, batch_size=100, collate_fn=collate_batch, pin_memory=True, num_workers=2)

test_dict = dict(sorted(test_dict.items(), key=lambda item: len(item[1])))
test_dataset = SequenceDataset(test_dict)
test_loader = DataLoader(test_dataset, batch_size=100, collate_fn=collate_batch, pin_memory=True, num_workers=2)

# Log training details
training_message = "Training on the whole train set"
print(training_message)  # Print to console
logging.info(training_message)  # Log to file

best_model, test_auc = train_dkt(
    model_params=best_config['models_params'],
    lr=best_config['learning_rate'],
    num_epochs=NUM_EPOCHS,
    device=device,
    train_loader=train_loader,
    val_loader=test_loader
)

# Log test AUC
test_auc_message = f"The achieved test AUC: {test_auc}"
print(test_auc_message)  # Print to console
logging.info(test_auc_message)  # Log to file

# Saving the best model
model_save_path = f"../data/models/best_dkt_model_embed_dim_{embed_dim}.pth"
torch.save(best_model.state_dict(), model_save_path)
model_save_message = f"Best model saved to {model_save_path}"
print(model_save_message)  # Print to console
logging.info(model_save_message)  # Log to file

# Save the best configuration
config_save_path = f"../data/models/best_dkt_config_embed_dim_{embed_dim}.json"
with open(config_save_path, "w") as f:
    json.dump(best_config, f, indent=4)

config_save_message = f"Best configuration saved to {config_save_path}"
print(config_save_message)  # Print to console
logging.info(config_save_message)  # Log to file


Training on the whole train set

Epoch 1



KeyboardInterrupt: 